# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/owendewing/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/owendewing/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [5]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [6]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [7]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [8]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [9]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [10]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [11]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [12]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '6f48f5'. Skipping!
Property 'summary' already exists in node '7f8ccc'. Skipping!
Property 'summary' already exists in node '3e20b8'. Skipping!
Property 'summary' already exists in node '67c9eb'. Skipping!
Property 'summary' already exists in node '12fc7f'. Skipping!
Property 'summary' already exists in node '0a73fb'. Skipping!
Property 'summary' already exists in node 'bfca27'. Skipping!
Property 'summary' already exists in node '8bbd04'. Skipping!
Property 'summary' already exists in node 'bdf404'. Skipping!
Property 'summary' already exists in node 'c581fd'. Skipping!
Property 'summary' already exists in node '1fa1e7'. Skipping!
Property 'summary' already exists in node 'ca2442'. Skipping!
Property 'summary' already exists in node 'a07953'. Skipping!
Property 'summary' already exists in node '66e7db'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '6f48f5'. Skipping!
Property 'summary_embedding' already exists in node '7f8ccc'. Skipping!
Property 'summary_embedding' already exists in node '0a73fb'. Skipping!
Property 'summary_embedding' already exists in node '67c9eb'. Skipping!
Property 'summary_embedding' already exists in node '66e7db'. Skipping!
Property 'summary_embedding' already exists in node '8bbd04'. Skipping!
Property 'summary_embedding' already exists in node '3e20b8'. Skipping!
Property 'summary_embedding' already exists in node 'bfca27'. Skipping!
Property 'summary_embedding' already exists in node '12fc7f'. Skipping!
Property 'summary_embedding' already exists in node 'c581fd'. Skipping!
Property 'summary_embedding' already exists in node '1fa1e7'. Skipping!
Property 'summary_embedding' already exists in node 'bdf404'. Skipping!
Property 'summary_embedding' already exists in node 'ca2442'. Skipping!
Property 'summary_embedding' already exists in node 'a07953'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 478)

We can save and load our knowledge graphs as follows.

In [13]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 478)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [15]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### ✅ Answer:

The Single Hop Specific Query Synthesizer is responsible for generating simple questions/queries that can be answered by reading just a single sentence or paragraph. In other words, only one retrieval is enough to answer the question.

The Multi Hop Specific Query Synthesizer generates multiple step questions. This means that the syntehsizer might need to grab information from different sources, but leads to one specific answer.

The Multi Hop Abstract Query Synthesizer also generates multiple step questions, but answers more conceptual questions. These questions require more higher-level reasoning.


Finally, we can use our `TestSetGenerator` to generate our testset!

In [16]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is Volume 2 about in the context of acade...,"[Chapter 1 Academic Years, Academic Calendars,...",Volume 2 covers information about academic yea...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(a) specify regarding th...,[Regulatory Citations Academic year minimums: ...,Regulatory Citations Academic year minimums: 3...,single_hop_specifc_query_synthesizer
2,How does Volume 8 address the inclusion of cli...,[Inclusion of Clinical Work in a Standard Term...,"Inclusion of clinical work in a standard term,...",single_hop_specifc_query_synthesizer
3,How does the Title IV program relate to paymen...,[Non-Term Characteristics A program that measu...,The context indicates that Title IV program di...,single_hop_specifc_query_synthesizer
4,What is a Direct Loen in the context of studen...,[both the credit or clock hours and the weeks ...,A Direct Loan is a type of federal student loa...,single_hop_specifc_query_synthesizer
5,How does inclusion of clinical work in a stand...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Inclusion of clinical work in standard term pe...,multi_hop_abstract_query_synthesizer
6,separate academic years for different program ...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The context explains that schools can define s...,multi_hop_abstract_query_synthesizer
7,How does the student complition of coursework ...,[<1-hop>\n\nboth the credit or clock hours and...,The student complition of coursework and instr...,multi_hop_abstract_query_synthesizer
8,Whch Volum 2 or 7 is imp for disbursmnt?,[<1-hop>\n\nboth the credit or clock hours and...,The context indicates that Volume 2 discusses ...,multi_hop_specific_query_synthesizer
9,hw do chptr 2 and 3 help with clincal work in ...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Chapter 2 discusses the inclusion of clinical ...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [17]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '3df70a'. Skipping!
Property 'summary' already exists in node '6758d8'. Skipping!
Property 'summary' already exists in node 'de8e97'. Skipping!
Property 'summary' already exists in node '4362b4'. Skipping!
Property 'summary' already exists in node 'f68e84'. Skipping!
Property 'summary' already exists in node '2f32f4'. Skipping!
Property 'summary' already exists in node 'f3ccf7'. Skipping!
Property 'summary' already exists in node 'd5d7af'. Skipping!
Property 'summary' already exists in node '7600d2'. Skipping!
Property 'summary' already exists in node 'e733a7'. Skipping!
Property 'summary' already exists in node '54b675'. Skipping!
Property 'summary' already exists in node 'fb8e15'. Skipping!
Property 'summary' already exists in node 'c39ef8'. Skipping!
Property 'summary' already exists in node '00acb7'. Skipping!


...

Property 'summary_embedding' already exists in node 'd5d7af'. Skipping!
Property 'summary_embedding' already exists in node 'f68e84'. Skipping!
Property 'summary_embedding' already exists in node '3df70a'. Skipping!
Property 'summary_embedding' already exists in node '2f32f4'. Skipping!
Property 'summary_embedding' already exists in node 'de8e97'. Skipping!
Property 'summary_embedding' already exists in node 'f3ccf7'. Skipping!
Property 'summary_embedding' already exists in node '6758d8'. Skipping!
Property 'summary_embedding' already exists in node '4362b4'. Skipping!
Property 'summary_embedding' already exists in node '54b675'. Skipping!
Property 'summary_embedding' already exists in node 'c39ef8'. Skipping!
Property 'summary_embedding' already exists in node '7600d2'. Skipping!
Property 'summary_embedding' already exists in node 'e733a7'. Skipping!
Property 'summary_embedding' already exists in node 'fb8e15'. Skipping!
Property 'summary_embedding' already exists in node '00acb7'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [18]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is the School Participation Division do w...,"[Chapter 1 Academic Years, Academic Calendars,...",The School Participation Division oversees the...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(a) specify regarding th...,[Regulatory Citations Academic year minimums: ...,Regulatory Citations Academic year minimums: 3...,single_hop_specifc_query_synthesizer
2,What is Volume 8 in relation to clinical work ...,[Inclusion of Clinical Work in a Standard Term...,"Volume 8, Chapter 3, provides guidance on exce...",single_hop_specifc_query_synthesizer
3,Is the Federal Work-Study (FWS) program subjec...,[Non-Term Characteristics A program that measu...,"No, the payment period is applicable to all Ti...",single_hop_specifc_query_synthesizer
4,How do the disbursement timing requirements fo...,[<1-hop>\n\nboth the credit or clock hours and...,Disbursement timing for federal financial aid ...,multi_hop_abstract_query_synthesizer
5,Waht are the diffrent academic year definision...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The context explains that every eligible progr...,multi_hop_abstract_query_synthesizer
6,How do disbursement timing requirements differ...,[<1-hop>\n\nboth the credit or clock hours and...,In clock-hour or non-term credit-hour programs...,multi_hop_abstract_query_synthesizer
7,disbursement timing in subscription programs a...,[<1-hop>\n\nboth the credit or clock hours and...,"In the context of subscription-based programs,...",multi_hop_abstract_query_synthesizer
8,How does Volume 8 inform the inclusion of clin...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Volume 8 provides guidance on including clinic...,multi_hop_specific_query_synthesizer
9,Can you tell me how Volume 2 and Volume 8 rela...,[<1-hop>\n\nInclusion of Clinical Work in a St...,"Based on the provided context, Volume 2 discus...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [19]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [20]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [21]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [22]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [23]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [24]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [25]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [26]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [27]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [28]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [29]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'Based on the provided context, the kinds of loans available include:\n\n- Direct Subsidized Loans (available only to undergraduate students)\n- Direct Unsubsidized Loans\n- Direct PLUS Loans (including student Federal PLUS Loans and parent PLUS Loans)\n- Subsidized and Unsubsidized Federal Stafford Loans (made under the FFEL Program before July 1, 2010)\n- Federal PLUS Loans (made under the FFEL Program before July 1, 2010)\n- Direct Consolidation Loans\n\nThese loans can be for preparatory coursework, teacher certification coursework, and standard eligible programs. Graduate or professional students are eligible only for Direct Unsubsidized Loans and Direct PLUS Loans, not Direct Subsidized Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [30]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [31]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

The qa_evaluator evaluates whether the generated answer is accurate given the question and the context/reference answer.

The labeled_helpfulness_evaluator uses an LLM to evaluate if the response is helpful to the user, given the context/correct answer. It uses the criteria to ensure that the model's output is helpful to the user _and_ that its aligned with the reference answer.

The empathy evaluator uses an LLM to evalulate if the response is empathetic. This can be used to ensure that the response doesn't sound robotic and that it acknowledges the user's concerns. this evaluator doesn't take into account the reference answer.

## LangSmith Evaluation

In [32]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'diligent-disease-68' at:
https://smith.langchain.com/o/d432d5de-9a60-4d76-b22f-5ee5c2cfdd2f/datasets/63cfdffd-3ab8-48f8-ae5d-ee27726c5c40/compare?selectedSessions=00b3704e-2437-4fcf-a414-57d64866a318




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Appendix A and Appendix B relate to the...,Appendix A provides examples related to non-te...,None,Appendix B provides detailed guidance on disbu...,1,1,0,3.268385,4e2658c8-db90-4afc-a152-3d7059b59ab8,b92a2de7-9448-47b4-93c6-e67b64160b79
1,Considering the detailed requirements outlined...,"Based on the provided context, an academic pro...",None,To ensure compliance with federal regulations ...,1,1,0,6.127068,f48ee5e3-3916-4b84-a6dd-07b6bc728ff0,3a761722-e2fa-41f9-a8a2-1050933bdbf8
2,Can you tell me how Volume 2 and Volume 8 rela...,I don't know.,None,"Based on the provided context, Volume 2 discus...",0,0,0,5.732577,44f7ad30-9c5b-424d-bbb4-f4355515f5e8,b2aabad9-d120-4116-a776-381db70a7fee
3,How does Volume 8 inform the inclusion of clin...,Volume 8 explains that periods of clinical wor...,None,Volume 8 provides guidance on including clinic...,1,1,0,9.521389,b177a933-8673-4f64-943e-7eeabe48a32d,87cc060b-3d04-4754-95c7-11cc48a35774
4,disbursement timing in subscription programs a...,Based on the provided context:\n\nIn subscript...,None,"In the context of subscription-based programs,...",1,1,0,4.706639,9fcff8a5-f9a9-4144-bb01-01b4ac5cf9b7,95b8f4b9-f31b-4be9-9a05-b4bd1312999f
5,How do disbursement timing requirements differ...,Based on the provided context:\n\n- **Subscrip...,None,In clock-hour or non-term credit-hour programs...,1,1,0,4.807064,d833e201-efc2-4caa-a81f-d20739876904,f75aa075-28ed-4f5a-a5cc-26871c64df46
6,Waht are the diffrent academic year definision...,"Based on the provided context, different acade...",None,The context explains that every eligible progr...,1,0,0,6.245324,d257596c-0f92-4ee0-89be-c06dfd5df842,31c5082c-50e7-42f0-b31a-cfaa56cbcbcc
7,How do the disbursement timing requirements fo...,Based on the provided context:\n\n**Disburseme...,None,Disbursement timing for federal financial aid ...,1,0,0,6.195346,3ab58132-aa1c-450b-a667-13ca43718ed3,50467006-5d66-4683-b6c9-c59f31d7a781
8,Is the Federal Work-Study (FWS) program subjec...,"No, the Federal Work-Study (FWS) program is no...",None,"No, the payment period is applicable to all Ti...",1,1,0,2.103682,480e183c-49d4-4e95-a960-b24f623c6f51,3f716cce-3cdb-4e3f-a569-ad7ea276780a
9,What is Volume 8 in relation to clinical work ...,I don't know.,None,"Volume 8, Chapter 3, provides guidance on exce...",0,0,0,0.733006,525bd67e-53e5-483d-aa40-809e09c97b46,6f8dfe2b-3717-4543-bdcb-d6b844985bdf


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [33]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [34]:
rag_documents = docs

In [35]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

##### ✅ Answer:

If the chunk size is too small, this could allow for super precise context; however, some important meaning and information might get split in between different chunks. The LLM also might miss important connections between ideas if they are split into different chunks. However, this chunk size might be ideal for the single hop synthesizer, which requires less context to answer a question.

If the chunk size is too large, this would help with preserving ideas; however, there can also be issues with this approach. This would lead to slower retrieval, and less precision from the answers returned. It also would increase the token count, which could lead to worse performance.

In [36]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

##### ✅ Answer:

Switching from text-embedding-3-small to text-embedding-3-large definitely changes the performance of our application. Text-embedding-3-large has more dimensions (default is 3072), meaning that it can allow for greater separation between different meanings and closer representation for similar meanings, allowing for more detailed and relevant retrieval. It can also detect more differences in tone and context, making it better for complex text/documents and multi-hop reasoning.

However, the text-embedding-3-small embedding model has a lower cost and faster search, which can be good in applications where speed is more important than precise retrieval.

In [37]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [38]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [40]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [41]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question—it's great that you're seeking clarity on the types of loans available. Based on the information provided, there are several kinds of Direct Loans you might consider:\n\n1. **Direct Subsidized Loans** – These loans are need-based, meaning you can borrow up to the amount of your financial need (calculated as Cost of Attendance minus other aid). Interest is paid by the government while you’re in school.\n\n2. **Direct Unsubsidized Loans** – These are available to both dependent and independent students regardless of financial need. Interest accrues while you’re in school.\n\n3. **Direct PLUS Loans** – These loans are for parents of dependent students or graduate/professional students themselves. Parents can borrow up to the student’s Cost of Attendance minus other aid, but the parent must meet PLUS eligibility requirements.\n\nIt’s also noted that if the parent is unable to get a Direct PLUS Loan for a dependent student, the student may qualify for additional

Finally, we can evaluate the new chain on the same test set!

In [42]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'only-nut-30' at:
https://smith.langchain.com/o/d432d5de-9a60-4d76-b22f-5ee5c2cfdd2f/datasets/63cfdffd-3ab8-48f8-ae5d-ee27726c5c40/compare?selectedSessions=e93ee366-c9d9-4510-bd08-61fef708ea7f




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do Appendix A and Appendix B relate to the...,Thank you for your thoughtful question. Based ...,None,Appendix B provides detailed guidance on disbu...,0,0,1,5.602923,4e2658c8-db90-4afc-a152-3d7059b59ab8,82ab4e83-5569-451e-9f95-50efd080a988
1,Considering the detailed requirements outlined...,Thank you for your thoughtful and detailed que...,None,To ensure compliance with federal regulations ...,1,1,1,11.769890,f48ee5e3-3916-4b84-a6dd-07b6bc728ff0,6d0d6524-4b96-4778-8dbd-24a31784f0f7
2,Can you tell me how Volume 2 and Volume 8 rela...,Thank you so much for your thoughtful question...,None,"Based on the provided context, Volume 2 discus...",0,0,1,5.727556,44f7ad30-9c5b-424d-bbb4-f4355515f5e8,4a90b2d6-670b-4387-a0b6-17deec81afe4
3,How does Volume 8 inform the inclusion of clin...,Thank you for your thoughtful question. Based ...,None,Volume 8 provides guidance on including clinic...,1,0,1,3.888776,b177a933-8673-4f64-943e-7eeabe48a32d,3a4bb5a6-baed-45c5-bdfd-4bfcdb6c19b8
4,disbursement timing in subscription programs a...,Thank you for your thoughtful question about d...,None,"In the context of subscription-based programs,...",1,1,1,24.880820,9fcff8a5-f9a9-4144-bb01-01b4ac5cf9b7,2b705348-d2e5-43eb-aef6-4228dc9d5f8c
5,How do disbursement timing requirements differ...,Thank you for your thoughtful question about d...,None,In clock-hour or non-term credit-hour programs...,1,0,1,9.184662,d833e201-efc2-4caa-a81f-d20739876904,b5276f11-74f5-4bcf-a522-24c45f684e3c
6,Waht are the diffrent academic year definision...,Thank you for your thoughtful question. From t...,None,The context explains that every eligible progr...,1,0,1,6.073044,d257596c-0f92-4ee0-89be-c06dfd5df842,01d09c1d-cefc-4126-a4c6-48b724ad4eb9
7,How do the disbursement timing requirements fo...,Thank you for your thoughtful question. It’s c...,None,Disbursement timing for federal financial aid ...,0,0,1,8.745800,3ab58132-aa1c-450b-a667-13ca43718ed3,8721a5cc-45fd-4858-b69f-cac117d047a7
8,Is the Federal Work-Study (FWS) program subjec...,Thank you for your thoughtful question. Based ...,None,"No, the payment period is applicable to all Ti...",1,1,1,2.545425,480e183c-49d4-4e95-a960-b24f623c6f51,8d721c8d-515d-402e-8b3f-0ecf2377eecb
9,What is Volume 8 in relation to clinical work ...,Thank you for your thoughtful question. Based ...,None,"Volume 8, Chapter 3, provides guidance on exce...",1,1,1,3.318269,525bd67e-53e5-483d-aa40-809e09c97b46,0cb2bc61-d245-4a09-8a24-e863c1762e46


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

![Evaluation](LangSmith_Evaluation.png)

1. Correctness slightly decreased from the first chain to the second chain, which I was slightly suprised about. I thought that the switch from small to large embeddings would allow the second chain to retrieve more relevant and accurate responses. However, a reason for this switch could be because the chunking size increased from the first chain to the second chain. A smaller chunk size could lead to more precise and accurate responses. Furthermore, another reason could have to do with the second chain's focus on empathy. This could lead to answers that focus on emotion and kindness that end up overpowering the accuracy and correctness a little bit.

2. Empathy significantly increased from the first chain to the second chain which is no suprise, because the second chain contained this sentence in the RAG prompt, "You must answer the question using empathy and kindness, and make sure the user feels heard." This heavily contributes to a stronger score on the empathy evaluator, which looks for responses that make the user feel seen.

3. Helpfulness also slightly decreased, which makes sense given that correctness slightly decreased. I think that this is also due to the smaller chunk size of the first chain and focus on empathy from the second chain.